# Installer le programme

Ce tutoriel vous invite à découvrir un agent artificiel basé sur le principe de la "Motivation Interactionnelle".

La Motivation Interactionnelle est un principe de conception d'agents artificiel  qui vise à donner l'impression que l'agent "aime" certaines interactions, et "n'aime pas" d'autres interactions.

C'est un observateur humain (vous) qui interprétez la motivation de l'agent en observant ses comportements.

In [ ]:
# @title Importer ColabTurtle
!pip3 install ColabTurtle
from ColabTurtle.Turtle import *

In [2]:
# @title Installer l'application
class Interaction:
    interaction_list = []

    def __init__(self, action, outcome, valence):
        self.action = action
        self.outcome = outcome
        self.valence = valence

    def __str__(self):
        """ Print interaction in the form <action><outcome>(<valence>) """
        return str(self.action) + str(self.outcome) + "(" + str(self.valence) + ")"

    def __hash__(self):
        """ The hash is necessary to use interactions as keys in a dictionary """
        return self.action * 10 + self.outcome

    def __eq__(self, other):
        """ Interactions are equal if they have the same action and the same outcome """
        if isinstance(other, self.__class__):
            return (self.action == other.action) and (self.outcome == other.outcome)
        else:
            return False

    @classmethod
    def create_or_retrieve(cls, action, outcome, valence=0):
        """ Use this methode to create a new interaction or to retrieve it if it already exists """
        interaction = Interaction(action, outcome, valence)

        if interaction in cls.interaction_list:
            i = cls.interaction_list.index(interaction)
            # print("Retrieving ", end="")
            # print(cls.interaction_list[i])
            return cls.interaction_list[i]
        else:
            # print("Creating ", end="")
            # print(interaction)
            cls.interaction_list.append(interaction)
            return interaction

class CompositeInteraction:
    composite_interaction_list = []

    def __init__(self, pre_interaction, post_interaction):
        self.pre_interaction = pre_interaction
        self.post_interaction = post_interaction
        self.weight = 1
        self.isActivated = False

    def increment_weight(self):
        self.weight += 1

    def __str__(self):
        """ Print interaction in the form <pre_interaction_post_interaction> """
        return "<" + self.pre_interaction.__str__() + self.post_interaction.__str__() + ">"

    def __hash__(self):
        """ The hash is necessary to use interactions as keys in a dictionary """
        return self.pre_interaction.__hash__()*100 + self.post_interaction.__hash__()

    def __eq__(self, other):
        """ Interactions are equal if they have the same pre and post interactions """
        if isinstance(other, self.__class__):
            return (self.pre_interaction == other.pre_interaction) and (self.post_interaction == other.post_interaction)
        else:
            return False

    @classmethod
    def create_or_retrieve(cls, pre_interaction, post_interaction):
        interaction = CompositeInteraction(pre_interaction, post_interaction)

        if interaction in cls.composite_interaction_list:
            i = cls.composite_interaction_list.index(interaction)
            # print("Retrieving ", end="")
            # print(cls.interaction_list[i])
            return cls.composite_interaction_list[i]
        else:
            # print("Creating ", end="")
            # print(interaction)
            cls.composite_interaction_list.append(interaction)
            return interaction

    @classmethod
    def create_or_reinforce(cls, pre_interaction, post_interaction):
        interaction = CompositeInteraction(pre_interaction, post_interaction)

        if interaction in cls.composite_interaction_list:
            i = cls.composite_interaction_list.index(interaction)
            print("reinforcing:", cls.composite_interaction_list[i].__str__())
            cls.composite_interaction_list[i].weight += 1
            return cls.composite_interaction_list[i]
        else:
            print("Learning:", interaction.__str__())
            cls.composite_interaction_list.append(interaction)
            return interaction

class Agent5:
    def __init__(self, _hedonist_table):
        """ Creating our agent """
        # These values give a nice demo with osoyoo car
        self.hedonist_table = _hedonist_table
        self._action = 0
        self.anticipated_outcome = None

        self.memory: list[CompositeInteraction] = []
        self.previous_interaction = None
        self.last_interaction = None

    def action(self, outcome):
        """ tracing the previous cycle """
        # if self._action is not None:
            # print("Action: " + str(self._action) +
            #       ", Anticipation: " + str(self.anticipated_outcome) +
            #       ", Outcome: " + str(outcome) +
            #       ", Satisfaction: (anticipation: " + str(self.anticipated_outcome == outcome) +
            #       ", valence: " + str(self.hedonist_table[self._action][outcome]) + ")")

        """ Recording previous experience """
        self.previous_interaction = self.last_interaction
        valence = self.hedonist_table[self._action][outcome]
        self.last_interaction = Interaction.create_or_retrieve(self._action, outcome, valence)
        # print("Enacted interaction ", end="")
        # print(self.last_interaction)
        if self.previous_interaction is not None:
            composite_interaction = CompositeInteraction.create_or_retrieve(self.previous_interaction,
                                                                            self.last_interaction)
            if composite_interaction not in self.memory:
                self.memory.append(composite_interaction)
                # print("Learning " + composite_interaction.__str__())
            else:
                i = self.memory.index(composite_interaction)
                self.memory[i].increment_weight()
                # print("Reinforcing " + self.memory[i].__str__())

        """ Selecting the next action to enact """
        self._action = 0
        self.anticipated_outcome = None
        proclivity_list = [0, 0, 0]
        # print(self.memory)
        if self.memory:
            activated_interactions = [ci for ci in self.memory if ci.pre_interaction == self.last_interaction]
            for ai in activated_interactions:
                proclivity_list[ai.post_interaction.action] += ai.weight * ai.post_interaction.valence

        # print("Proclivity list: ", end="")
        # print(proclivity_list)
        max_proclivity = max(proclivity_list)
        self._action = proclivity_list.index(max_proclivity)

        """ Computing the anticipation """  # TODO:improve this
        if Interaction.interaction_list:
            for ai in Interaction.interaction_list:
                if ai.action == self._action:
                    if max_proclivity > 0 and ai.valence > 0:
                        self.anticipated_outcome = ai.outcome
                    if max_proclivity < 0 and ai.valence < 0:
                        self.anticipated_outcome = ai.outcome

        return self._action

BORDER_WIDTH = 20

class ColabTurtleEnacter:

    def __init__(self):
        """ Creating the Turtle window """
        bgcolor("lightGray")
        penup()
        goto(window_width() / 2, window_height()/2)
        face(0)
        pendown()
        color("green")

    def outcome(self, action):
        """ Enacting an action and returning the outcome """
        _outcome = 0
        for i in range(10):
            # _outcome = 0
            if action == 0:
                # move forward
                forward(10)
            elif action == 1:
                # rotate left
                left(4)
                forward(2)
            elif action == 2:
                # rotate right
                right(4)
                forward(2)

            # Bump on screen edge and return outcome 1
            if xcor() < BORDER_WIDTH:
                goto(BORDER_WIDTH, ycor())
                _outcome = 1
            if xcor() > window_width() - BORDER_WIDTH:
                goto(window_width() - BORDER_WIDTH, ycor())
                _outcome = 1
            if ycor() < BORDER_WIDTH:
                goto(xcor(), BORDER_WIDTH)
                _outcome = 1
            if ycor() > window_height() - BORDER_WIDTH:
                goto(xcor(), window_height() -BORDER_WIDTH)
                _outcome = 1

            # Change color
            if _outcome == 0:
                color("green")
            else:
                # Finit l'interaction
                color("red")
                # if action == 0:
                #     break
                if action == 1:
                    for j in range(10):
                        left(4)
                elif action == 2:
                    for j in range(10):
                        right(4)
                break

        return _outcome

# Définir les préférences de la tortue

La tortue peut faire trois actions: avancer, tourner à gauche, ou tourner à droite. Chacune de ces actions peut provoquer ou non une collision contre un mur de la fenêtre. Les combinaisons de ces trois actions et de ces deux résultats donnent six "interactions" possibles.

Vous définissez les "préférences" de chacune des six interactions possibles.
Une préférence positive signifie que la tortue "aime" cette interaction.
Une préférence négative signifient que la tortue "n'aime pas" cette interaction.

In [3]:
avancer = 4
avancer_contre_un_mur = -10
tourner_a_gauche = -5
tourner_a_gauche_contre_un_mur = -5
tourner_a_droite = -1
tourner_a_droite_contre_un_mur = -1

# Lancer l'expérimentation

Choisir le nombre de cycles d'interaction

In [4]:
nombre_de_cycles = 50

In [7]:
# @title Executer l'experimentation
initializeTurtle()

bgcolor("lightGray")
penup()
goto(window_width() / 2, window_height()/2)
face(0)
pendown()
color("green")
speed(10)

valences = [[avancer, avancer_contre_un_mur], [tourner_a_gauche, tourner_a_gauche_contre_un_mur], [tourner_a_droite, tourner_a_droite_contre_un_mur]]

Interaction.interaction_list = []
CompositeInteraction.composite_interaction_list = []

a = Agent5(valences)
# a = AgenteTortugaInteligente(valences)
e = ColabTurtleEnacter()

outcome = 0

for i in range(nombre_de_cycles):
    action = a.action(outcome)
    outcome = e.outcome(action)

# Exercice

Changes les préférences des interactions et relancez l'expérimentation.

Pour commencer, donner une préférence positive à seulement une interaction, et des préférences négatives à toute les autres. Par exemple l'interaction avancer_contre_un_mur ou l'interaction tourner_a_gauche.
